#

# 📊 NOTEBOOK 1: COLETA E PREPARAÇÃO DE DADOS
**Análise de Sentimento em Decisões Judiciais - TJCE**

## 🔧 1. CONFIGURAÇÃO DO AMBIENTE

### 1.1 Instalação de Dependências
**Descrição**: Instalação das bibliotecas Python necessárias.

In [3]:
# Instalar bibliotecas necessárias
!pip install -r requirements.txt

### 1.2 Importação de Bibliotecas

In [4]:
import pandas as pd
import json
from collections import Counter
import requests
import csv

## 📥 2. COLETA DE DADOS VIA API CNJ *DATAJUD*


### 2.1 Configuração da API e Parâmetros de Busca

 **Descrição**: Configuração da requisição à API pública do CNJ. O código de assunto 12487 refere-se a "Fornecimento de medicamentos", encontrado no site oficial do CNJ (https://www.cnj.jus.br/sgt/consulta_publica_assuntos.php).

 Para encontrar o código no site do cnj:
   1. Procure por "Medicamentos"
   2. Selecione a 4ª opção: `Fornecimento de medicamentos`
   3. Verifique o código que aparece, `12487 Fornecimento de medicamentos`

In [5]:
url = "https://api-publica.datajud.cnj.jus.br/api_publica_tjsc/_search"
api_key = "APIKey cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="

payload = json.dumps({
    "size": 10000,
    "query": {"match": {"assuntos.codigo": "12487"}},  # Código: Fornecimento de medicamentos
    "sort": [{"dataAjuizamento": {"order": "desc"}}]
})

headers = {
    'Authorization': api_key,
    'Content-Type': 'application/json'
}

### 2.2 Requisição e Validação dos Dados
**Descrição**: Execução da requisição HTTP POST para a API e validação inicial do volume de dados retornados.

In [6]:
response = requests.request("POST", url, headers=headers, data=payload)
dados_dict = response.json()
print(f"Total de processos encontrados na API: {dados_dict['hits']['total']['value']}")
print(f"Processos retornados nesta consulta: {len(dados_dict['hits']['hits'])}")

Total de processos encontrados na API: 6853
Processos retornados nesta consulta: 6853


## 🔄 3. PROCESSAMENTO E ESTRUTURAÇÃO DOS DADOS

### 3.1 Extração de Campos Essenciais
**Descrição**: Extração dos campos essenciais de cada processo (número, grau de jurisdição, data de ajuizamento e movimentos processuais) e criação do DataFrame principal.

In [8]:
processos = []

for hit in dados_dict['hits']['hits']:
    processo = hit['_source']

    numero_processo = processo['numeroProcesso']
    grau = processo['grau']
    # Use .get() to safely access 'dataAjuizamento', returning None if not found
    data_ajuizamento = processo.get('dataAjuizamento')
    movimentos = processo.get('movimentos', [])

    processos.append([
        numero_processo,
        grau,
        data_ajuizamento,
        movimentos
    ])

df = pd.DataFrame(
    processos,
    columns=[
        'numero_processo',
        'grau',
        'data_ajuizamento',
        'movimentos'
    ]
)

print(f"Total de processos no DataFrame: {len(df)}")
df.sample(5)

Total de processos no DataFrame: 6853


,numero_processo,grau,data_ajuizamento,movimentos
3268,50257766820238240000,G2,2023-04-28T15:47:54.000Z,"[{'complementosTabelados': [{'codigo': 18, 'va..."
5840,50136727720208240023,G1,2020-02-14T17:54:52.000Z,"[{'complementosTabelados': [{'codigo': 4, 'val..."
3222,50078211320238240036,G1,2023-05-25T15:31:29.000Z,"[{'complementosTabelados': [{'codigo': 19, 'va..."
2794,50085621820238240080,G1,2023-12-11T10:56:47.000Z,"[{'codigo': 1051, 'nome': 'Decurso de Prazo', ..."
4863,50048883220218240038,G1,2021-02-09T15:33:33.000Z,"[{'codigo': 848, 'nome': 'Trânsito em julgado'..."


## 🔍 4. IDENTIFICAÇÃO E ANÁLISE DE DECISÕES

### 4.1 Definição de Termos de Decisão
**Descrição**: Definição dos termos-chave que caracterizam decisões judiciais relevantes para a análise.

In [9]:
from collections import Counter

termos_decisao = [
    "Procedência",
    "Improcedência",
    "Improcedência do pedido e improcedência do pedido contraposto",
    "Procedência do pedido e procedência do pedido contraposto",
    "Deferido",
    "Indeferido",
]

### 4.2 Extração de Decisões dos Movimentos Processuais
**Descrição**: Busca sistemática nos movimentos processuais por termos de decisão.

**Importante**: Filtra apenas processos de primeira instância (grau G1) para garantir homogeneidade na análise.

In [10]:
decisoes_por_processo = []
tipos_decisao_contagem = []

for _, row in df.iterrows():
    numero = row['numero_processo']
    movimentos = row['movimentos']
    grau = row['grau']
    data_ajuizamento = row['data_ajuizamento']

    decisoes_encontradas = []

    if movimentos:
        for mov in movimentos:
            nome_mov = mov.get('nome', '')

            # Filtrar apenas decisões de primeira instância (G1)
            if any(termo in nome_mov for termo in termos_decisao) and grau == 'G1':
                decisoes_encontradas.append(nome_mov)
                tipos_decisao_contagem.append(nome_mov)

    if decisoes_encontradas:
        decisoes_por_processo.append({
            'numero_processo': numero,
            'data_ajuizamento': data_ajuizamento,
            'decisoes': decisoes_encontradas,
            'grau': grau
        })

print(f"Processos com decisões: {len(decisoes_por_processo)} de {len(df)}")
print("\nTipos de decisões encontradas:")
for tipo, count in Counter(tipos_decisao_contagem).most_common(10):
    print(f"  {tipo}: {count}")

Processos com decisões: 1085 de 6853

Tipos de decisões encontradas:
  Procedência: 620
  Improcedência: 248
  Procedência em Parte: 232
  Procedência do Pedido - Reconhecimento pelo réu: 7


### 4.3 Criação de DataFrame de Decisões
**Descrição**: Criação de um DataFrame específico para decisões, removendo duplicatas por processo (mantém apenas a primeira decisão encontrada).

In [11]:
decisoes_lista = []

for item in decisoes_por_processo:
    for decisao in item['decisoes']:
        decisoes_lista.append({
            'numero_processo': item['numero_processo'],
            'tipo_decisao': decisao,
            'grau': item['grau'],
            'data_ajuizamento': item['data_ajuizamento']
        })

df_decisoes = pd.DataFrame(decisoes_lista)
df_decisoes = df_decisoes.drop_duplicates(subset='numero_processo', keep='first')
df_decisoes.head(10)

,numero_processo,tipo_decisao,grau,data_ajuizamento
0,50542077220258240023,Procedência,G1,20250825190649
1,50508473220258240023,Procedência,G1,20250807150347
2,50075127220258240019,Procedência,G1,20250805182109
3,50016074720258240032,Procedência em Parte,G1,20250731141216
4,50136040620258240039,Procedência,G1,20250725094915
5,50009980520258240084,Procedência,G1,20250714155959
6,50427894020258240023,Procedência,G1,20250625125025
7,50097562020258240036,Improcedência,G1,20250618085036
8,50104403320258240039,Procedência em Parte,G1,20250609191311
9,50052955620258240019,Procedência,G1,20250529180016


## 📊 5. SEGMENTAÇÃO POR TIPO DE DECISÃO

### 5.1 Separação por Categorias de Decisão
**Descrição**: Segmentação do dataset em categorias de decisão para análise comparativa posterior.

In [12]:
# Separar decisões por tipo
df_procedencia = df_decisoes[df_decisoes['tipo_decisao'] == 'Procedência'].copy()
df_improcedencia = df_decisoes[df_decisoes['tipo_decisao'] == 'Improcedência'].copy()

df_decisoes_completo = pd.concat([
    df_procedencia,
    df_improcedencia,
])

print(f"\n=== TODOS OS REGISTROS COM DECISÃO ===")
print(f"Total de registros: {len(df_decisoes_completo)}")
print(f"\nDistribuição por tipo:")
print(f"  - Procedência: {len(df_procedencia)}")
print(f"  - Improcedência: {len(df_improcedencia)}")

df = df_decisoes_completo

df


=== TODOS OS REGISTROS COM DECISÃO ===
Total de registros: 800

Distribuição por tipo:
  - Procedência: 569
  - Improcedência: 231


,numero_processo,tipo_decisao,grau,data_ajuizamento
0,50542077220258240023,Procedência,G1,20250825190649
1,50508473220258240023,Procedência,G1,20250807150347
2,50075127220258240019,Procedência,G1,20250805182109
4,50136040620258240039,Procedência,G1,20250725094915
5,50009980520258240084,Procedência,G1,20250714155959
...,...,...,...,...
1055,03146926820178240008,Improcedência,G1,2017-09-18T13:53:48.000Z
1056,03125794420178240008,Improcedência,G1,2017-08-09T14:40:45.000Z
1069,03003242920168240060,Improcedência,G1,2016-06-16T10:46:56.000Z
1081,03006377320158240076,Improcedência,G1,2015-06-15T22:02:21.000Z


## 🧹 6. LIMPEZA E CONSOLIDAÇÃO DO DATASET

### 6.1 Remoção de Valores Nulos

In [13]:
# Removendo registros NaN
df = df.dropna()
display(df.head(4))

,numero_processo,tipo_decisao,grau,data_ajuizamento
0,50542077220258240023,Procedência,G1,20250825190649
1,50508473220258240023,Procedência,G1,20250807150347
2,50075127220258240019,Procedência,G1,20250805182109
4,50136040620258240039,Procedência,G1,20250725094915


### 6.2 Reset de Índice

In [14]:
# Resetando index para manter organização do dataframe
df = df.reset_index(drop=True)
df

,numero_processo,tipo_decisao,grau,data_ajuizamento
0,50542077220258240023,Procedência,G1,20250825190649
1,50508473220258240023,Procedência,G1,20250807150347
2,50075127220258240019,Procedência,G1,20250805182109
3,50136040620258240039,Procedência,G1,20250725094915
4,50009980520258240084,Procedência,G1,20250714155959
...,...,...,...,...
795,03146926820178240008,Improcedência,G1,2017-09-18T13:53:48.000Z
796,03125794420178240008,Improcedência,G1,2017-08-09T14:40:45.000Z
797,03003242920168240060,Improcedência,G1,2016-06-16T10:46:56.000Z
798,03006377320158240076,Improcedência,G1,2015-06-15T22:02:21.000Z


## 📅 7. EXTRAÇÃO DE FEATURES TEMPORAIS

### 7.1 Extração de Dia da Semana e Ano
**Descrição**: Criação de features temporais (dia da semana e ano) para análise de padrões temporais nas decisões judiciais.

In [15]:
import pandas as pd

# Converter data_ajuizamento para datetime
df['data_ajuizamento_dt'] = pd.to_datetime(
    df['data_ajuizamento'],
    format='mixed', # Alterado para 'mixed' para lidar com diferentes formatos de data
    errors='coerce' # Adicionado para converter erros de parsing para NaT
)

# Remover linhas onde a conversão da data falhou, resultando em NaT
df.dropna(subset=['data_ajuizamento_dt'], inplace=True)

# Garantir que a coluna 'data_ajuizamento_dt' tenha o tipo de dado datetime64[ns]
df['data_ajuizamento_dt'] = pd.to_datetime(df['data_ajuizamento_dt'], utc=True)

dias_semana = {0: "Segunda", 1: "Terça", 2: "Quarta", 3: "Quinta", 4: "Sexta"}

# Extrair dia da semana (segunda = 0 | terça = 1 | ... | sexta = 4)
df['dia_semana'] = df['data_ajuizamento_dt'].dt.weekday.map(dias_semana)
df['ano'] = df['data_ajuizamento_dt'].dt.year

# Excluir coluna intermediária
df = df.drop(columns=['data_ajuizamento', 'data_ajuizamento_dt'])

df

/tmp/ipython-input-838281323.py:4: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['data_ajuizamento_dt'] = pd.to_datetime(


,numero_processo,tipo_decisao,grau,dia_semana,ano
0,50542077220258240023,Procedência,G1,Segunda,2025
1,50508473220258240023,Procedência,G1,Quinta,2025
2,50075127220258240019,Procedência,G1,Terça,2025
3,50136040620258240039,Procedência,G1,Sexta,2025
4,50009980520258240084,Procedência,G1,Segunda,2025
...,...,...,...,...,...
795,03146926820178240008,Improcedência,G1,Segunda,2017
796,03125794420178240008,Improcedência,G1,Quarta,2017
797,03003242920168240060,Improcedência,G1,Quinta,2016
798,03006377320158240076,Improcedência,G1,Segunda,2015


### 7.2 Verificando ano mínimo e máximo do conjunto de dados

In [16]:
min = df['ano'].min()
max = df['ano'].max()

print(f"Ano mínimo: {min}")
print(f"Ano máximo: {max}")

Ano mínimo: 2006
Ano máximo: 2025


## 💾 8. EXPORTAÇÃO DE DADOS INTERMEDIÁRIOS

### 8.1 Exportação para CSV
**Descrição**: Exportação dos dados processados para uso posterior:

*  processos_decisao_dia_ano.csv: Dataset completo com decisões e features temporais
*  numeros_processos.csv: Lista única de números de processo para scraping de PDFs




In [17]:
df.to_csv("processos_dia_ano.csv", sep=';', index=False)
df['numero_processo'].drop_duplicates().to_csv("numeros_processos.csv", index=False)

print("\n=== ARQUIVOS EXPORTADOS ===")
print("  - processos_dia_ano.csv")
print("  - numeros_processos.csv")


=== ARQUIVOS EXPORTADOS ===
  - processos_dia_ano.csv
  - numeros_processos.csv


## 📄 9. INSTRUÇÕES PARA COLETA DE DOCUMENTOS COMPLETOS

### 9.1 Scraping de PDFs (Executar Localmente)
**Pré-requisitos:**
1. Clone o repositório: `git clone https://github.com/GiovanniBrigido/trabalho-final-deep-learning.git`
2. Navegue até o diretório do projeto
3. Execute os seguintes comandos:

**Comandos:**
```bash
python -m venv venv
pip install -r requirements.txt
playwright install chromium
python ./scraper_pdf_tjce.py
```

O que o script faz:

*   Lê o arquivo numeros_processos.csv
*   Acessa o site do TJCE
*   Baixa os PDFs das sentenças completas
*   Salva os arquivos na pasta data/decisoes





### 9.2 Extração de Texto dos PDFs (Executar Localmente)
Após gerar os arquivos básicos acima, execute o script `scraper_pdf_tjce.py` para realizar a extração de PDF's.

**Execute:**
```bash
python ./extrair_decisoes.py
```
**O que o script faz**:

* Lê todos os PDFs baixados
* Extrai o texto completo de cada sentença com a bib PyPDF2
* Gera o arquivo decisoes_extraidas.csv com:
   * Número do processo
   * Texto completo da decisão
   * Metadados adicionais

**Arquivo gerado:**

*  decisoes_extraidas.csv
* **!! SERÁ USADO NO NOTEBOOK 2 !!**